## 9. Підготовка даних для Tableau (Data Preparation for Tableau)

In [1]:
# Імпортуємо необхідні бібліотеки

import pandas as pd
import numpy as np
import json
from pathlib import Path

In [2]:
# Знаходимо папку з обробленими даними

current_dir = Path.cwd()

possible_paths = [
    parent / "data_processed"
    for parent in [current_dir, *current_dir.parents]
]

processed_path = next(
    (path for path in possible_paths if path.is_dir()),
    None
)

if processed_path is None:
    raise FileNotFoundError(
        "Не знайдено папку data_processed"
    )


# Визначаємо назви датасетів

dataset_names = [
    "01_landing_page",
    "02_authenticated_create_room",
    "02_guest_create_room",
    "03_authenticated_host_room_invite_friends",
    "03_guest_host_room_invite_friends",
    "06_authenticated_swipe_session",
    "06_guest_swipe_session",
    "08_authenticated_results_winners",
    "08_guest_results_winners",
    "09_sign_in",
    "10_sign_up",
    "11_home_signed_in",
    "12_settings"
]


# Завантажуємо датасети

dataframes = {}

for name in dataset_names:

    file_path = processed_path / f"{name}.csv"

    if not file_path.exists():
        raise FileNotFoundError(
            f"Не знайдено файл: {file_path}"
        )

    df = pd.read_csv(
        file_path,
        low_memory=False
    )

    # Додаємо назву джерела даних

    df["source_dataset"] = name

    dataframes[name] = df


# Об'єднуємо всі датасети

all_events = pd.concat(
    dataframes.values(),
    ignore_index=True
)


# Відновлюємо типи ідентифікаторів

id_columns = [
    "event_id",
    "page_view_id",
    "visitor_id",
    "user_id",
    "room_id",
    "participant_id",
    "room_title_id",
    "title_id",
    "archive_session_id"
]

for column in id_columns:

    if column in all_events.columns:
        all_events[column] = all_events[column].astype("string")


# Перетворюємо часові значення

if "event_timestamp" in all_events.columns:

    all_events["event_timestamp"] = pd.to_datetime(
        all_events["event_timestamp"],
        errors="coerce",
        utc=True
    )


# Перетворюємо числові колонки

numeric_columns = [
    "card_position",
    "vote_average",
    "release_year",
    "runtime_minutes"
]

for column in numeric_columns:

    if column in all_events.columns:

        all_events[column] = pd.to_numeric(
            all_events[column],
            errors="coerce"
        )


# Відновлюємо Boolean-значення

if "event_value_bool" in all_events.columns:

    bool_values = all_events["event_value_bool"].astype("object")

else:

    bool_values = pd.Series(
        pd.NA,
        index=all_events.index,
        dtype="object"
    )

if "event_value" in all_events.columns:

    event_values = all_events["event_value"].astype("object")

    bool_values = bool_values.where(
        pd.notna(bool_values),
        event_values
    )

all_events["event_value_bool"] = (
    bool_values
    .astype("string")
    .str.strip()
    .str.lower()
    .map({
        "true": True,
        "false": False
    })
    .astype("boolean")
)


# Відокремлюємо фактичні події

if "event_value_type" not in all_events.columns:

    raise ValueError(
        "Відсутня колонка event_value_type"
    )

is_boolean = (
    all_events["event_value_type"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("boolean")
)

is_true = (
    all_events["event_value_bool"]
    .eq(True)
    .fillna(False)
)

activity_df = all_events.loc[
    (~is_boolean) | is_true
].copy()


# Перевіряємо завантажені дані

print("Кількість датасетів:", len(dataframes))
print("Загальна кількість записів:", f"{len(all_events):,}")
print("Кількість фактичних подій:", f"{len(activity_df):,}")

print("\nКількість записів за джерелами:")

display(
    activity_df["source_dataset"]
    .value_counts()
    .rename_axis("dataset")
    .reset_index(name="records")
)

Кількість датасетів: 13
Загальна кількість записів: 69,431
Кількість фактичних подій: 47,657

Кількість записів за джерелами:


,dataset,records
0,06_guest_swipe_session,21230
1,06_authenticated_swipe_session,16170
2,01_landing_page,2083
3,08_authenticated_results_winners,1475
4,08_guest_results_winners,1397
5,02_guest_create_room,1375
6,02_authenticated_create_room,900
7,11_home_signed_in,822
8,03_guest_host_room_invite_friends,794
9,03_authenticated_host_room_invite_friends,666


In [3]:
# Визначаємо позитивні та негативні події голосування

vote_mapping = {
    "swipe_right": True,
    "Like_button": True,
    "movie_liked": True,
    "swipe_left": False,
    "dislike_button": False,
    "movie_disliked": False
}


# Відбираємо події голосування

vote_df = activity_df.loc[
    activity_df["event_name"].isin(vote_mapping.keys())
].copy()

if vote_df.empty:
    raise ValueError(
        "Не знайдено подій голосування"
    )


# Перевіряємо необхідні колонки

required_columns = [
    "event_id",
    "room_id",
    "participant_id",
    "user_type",
    "title_id"
]

missing_columns = [
    column
    for column in required_columns
    if column not in vote_df.columns
]

if missing_columns:
    raise ValueError(
        f"Відсутні необхідні колонки: {missing_columns}"
    )


# Додаємо колонки для додаткового аналізу

optional_columns = [
    "vote_average",
    "genres_json",
    "room_title_id"
]

for column in optional_columns:

    if column not in vote_df.columns:
        vote_df[column] = pd.NA


# Визначаємо тип голосу

vote_df["is_liked"] = (
    vote_df["event_name"]
    .map(vote_mapping)
    .astype(bool)
)


# Видаляємо записи без ключових ідентифікаторів

vote_df = vote_df.dropna(
    subset=[
        "event_id",
        "room_id",
        "participant_id",
        "user_type",
        "title_id"
    ]
).copy()


# Стандартизуємо тип користувача

vote_df["user_type"] = (
    vote_df["user_type"]
    .astype("string")
    .str.strip()
    .str.upper()
)


# Перетворюємо рейтинг фільму на числовий тип

vote_df["vote_average"] = pd.to_numeric(
    vote_df["vote_average"],
    errors="coerce"
)


# Перевіряємо дублікати голосів

if vote_df["room_title_id"].notna().all():

    vote_key_column = "room_title_id"

else:

    vote_key_column = "title_id"

duplicate_votes = vote_df.duplicated(
    subset=[
        "room_id",
        "participant_id",
        vote_key_column
    ],
    keep=False
)

if duplicate_votes.any():

    raise ValueError(
        "Виявлено дублікати голосів у межах сесій учасників"
    )


# Перевіряємо унікальність event_id

if vote_df["event_id"].duplicated().any():

    raise ValueError(
        "Виявлено повторні event_id у голосуванні"
    )


# Розраховуємо оцінки фільмів за типом голосу

vote_df["liked_movie_rating"] = vote_df[
    "vote_average"
].where(vote_df["is_liked"])

vote_df["disliked_movie_rating"] = vote_df[
    "vote_average"
].where(~vote_df["is_liked"])


# Формуємо сесії учасників

session_keys = [
    "room_id",
    "participant_id"
]


# Перевіряємо узгодженість типу користувача в межах сесії

user_type_check = (
    vote_df.groupby(session_keys)["user_type"]
    .nunique()
)

if (user_type_check > 1).any():

    raise ValueError(
        "В одній сесії учасника знайдено декілька типів користувачів"
    )


# Розраховуємо показники голосування для кожної сесії

session_summary = (
    vote_df.groupby(
        session_keys,
        as_index=False
    )
    .agg(
        user_type=("user_type", "first"),
        session_total_votes=("event_id", "count"),
        session_likes=("is_liked", "sum"),
        session_unique_movies=("title_id", "nunique"),
        session_avg_movie_rating=("vote_average", "mean"),
        session_avg_rating_liked=("liked_movie_rating", "mean"),
        session_avg_rating_disliked=("disliked_movie_rating", "mean")
    )
)


# Розраховуємо кількість негативних голосів

session_summary["session_dislikes"] = (
    session_summary["session_total_votes"] -
    session_summary["session_likes"]
)


# Розраховуємо частки позитивних і негативних голосів

session_summary["session_like_rate_pct"] = (
    session_summary["session_likes"] /
    session_summary["session_total_votes"] * 100
)

session_summary["session_dislike_rate_pct"] = (
    session_summary["session_dislikes"] /
    session_summary["session_total_votes"] * 100
)


# Розраховуємо кількість унікальних жанрів у кожній сесії

def parse_genres(value):

    if isinstance(value, (list, tuple)):
        return list(value)

    if value is None or pd.isna(value):
        return []

    try:
        parsed = json.loads(str(value))

        if isinstance(parsed, list):
            return parsed

        return []

    except (json.JSONDecodeError, TypeError):
        return []


genre_data = vote_df[
    [
        "event_id",
        "room_id",
        "participant_id",
        "genres_json"
    ]
].copy()

genre_data["genre"] = genre_data["genres_json"].apply(
    parse_genres
)

genre_data = genre_data.explode("genre")

genre_data["genre"] = (
    genre_data["genre"]
    .astype("string")
    .str.strip()
)

genre_data = genre_data.dropna(
    subset=["genre"]
).copy()

genre_data = genre_data.loc[
    genre_data["genre"] != ""
].copy()

genre_data = genre_data.drop_duplicates(
    subset=["event_id", "genre"]
)

session_genres = (
    genre_data.groupby(session_keys)["genre"]
    .nunique()
    .reset_index(name="session_unique_genres")
)

session_summary = session_summary.merge(
    session_genres,
    on=session_keys,
    how="left",
    validate="one_to_one"
)

session_summary["session_unique_genres"] = (
    session_summary["session_unique_genres"].fillna(0)
)


# Визначаємо поведінковий сегмент учасника

def define_voting_segment(like_rate):

    if pd.isna(like_rate):
        return pd.NA

    if like_rate >= 65:
        return "Переважно позитивні"

    elif like_rate < 45:
        return "Переважно негативні"

    else:
        return "Збалансовані"


session_summary["behavior_segment"] = (
    session_summary["session_like_rate_pct"]
    .apply(define_voting_segment)
)


# Перевіряємо результати сегментації

print("Кількість голосів:", f"{len(vote_df):,}")

print(
    "Кількість сесій учасників:",
    len(session_summary)
)

print("\nРозподіл поведінкових сегментів:")

display(
    session_summary["behavior_segment"]
    .value_counts()
    .rename_axis("behavior_segment")
    .reset_index(name="participant_sessions")
)

display(session_summary.head())

Кількість голосів: 34,000
Кількість сесій учасників: 1700

Розподіл поведінкових сегментів:


,behavior_segment,participant_sessions
0,Збалансовані,1010
1,Переважно позитивні,429
2,Переважно негативні,261


,room_id,participant_id,user_type,session_total_votes,session_likes,session_unique_movies,session_avg_movie_rating,session_avg_rating_liked,session_avg_rating_disliked,session_dislikes,session_like_rate_pct,session_dislike_rate_pct,session_unique_genres,behavior_segment
0,500001.0,700001.0,AUTHENTICATED,20,7,20,7.081,7.282857,6.972308,13,35.0,65.0,13,Переважно негативні
1,500001.0,700002.0,GUEST,20,11,20,7.081,7.441818,6.640000,9,55.0,45.0,13,Збалансовані
2,500001.0,700003.0,GUEST,20,12,20,7.081,6.726667,7.612500,8,60.0,40.0,13,Збалансовані
3,500001.0,700004.0,GUEST,20,9,20,7.081,7.100000,7.065455,11,45.0,55.0,13,Збалансовані
4,500001.0,700005.0,GUEST,20,8,20,7.081,6.176250,7.684167,12,40.0,60.0,13,Переважно негативні


In [4]:
# Створюємо фінальну таблицю на основі фактичних подій

tableau_df = activity_df.copy()


# Додаємо ознаку голосування

tableau_df["is_vote_event"] = (
    tableau_df["event_name"].isin(vote_mapping.keys())
)


# Додаємо позитивні та негативні голоси

tableau_df["is_liked"] = (
    tableau_df["event_name"]
    .map(vote_mapping)
    .astype("boolean")
)

tableau_df["vote_type"] = (
    tableau_df["event_name"]
    .map({
        "swipe_right": "LIKE",
        "Like_button": "LIKE",
        "movie_liked": "LIKE",
        "swipe_left": "DISLIKE",
        "dislike_button": "DISLIKE",
        "movie_disliked": "DISLIKE"
    })
    .astype("string")
)


# Створюємо числові показники для Tableau

tableau_df["vote_count"] = (
    tableau_df["is_vote_event"].astype(int)
)

tableau_df["like_count"] = (
    tableau_df["is_liked"].eq(True).fillna(False).astype(int)
)

tableau_df["dislike_count"] = (
    tableau_df["is_liked"].eq(False).fillna(False).astype(int)
)


# Створюємо унікальний ідентифікатор сесії учасника

valid_session_keys = (
    tableau_df["room_id"].notna() &
    tableau_df["participant_id"].notna()
)

tableau_df["participant_session_id"] = pd.Series(
    pd.NA,
    index=tableau_df.index,
    dtype="string"
)

tableau_df.loc[
    valid_session_keys,
    "participant_session_id"
] = (
    tableau_df.loc[valid_session_keys, "room_id"].astype("string") +
    "_" +
    tableau_df.loc[valid_session_keys, "participant_id"].astype("string")
)


# Визначаємо додаткові показники сесії

session_columns = [
    "room_id",
    "participant_id",
    "session_total_votes",
    "session_likes",
    "session_dislikes",
    "session_like_rate_pct",
    "session_dislike_rate_pct",
    "session_unique_movies",
    "session_unique_genres",
    "session_avg_movie_rating",
    "session_avg_rating_liked",
    "session_avg_rating_disliked",
    "behavior_segment"
]

session_data = session_summary[session_columns].copy()


# Додаємо показники сесій до загальної таблиці подій

tableau_df = tableau_df.merge(
    session_data,
    on=["room_id", "participant_id"],
    how="left",
    validate="many_to_one"
)


# Перетворюємо назви жанрів у зручний для читання формат

if "genres_json" in tableau_df.columns:

    tableau_df["genres_text"] = (
        tableau_df["genres_json"]
        .apply(parse_genres)
        .apply(lambda genres: " | ".join(map(str, genres)))
        .astype("string")
    )


# Перевіряємо кількість записів після об'єднання

print("Кількість фактичних подій:", len(activity_df))

print("Кількість записів після об'єднання:", len(tableau_df))

print(
    "Кількість записів голосування:",
    int(tableau_df["vote_count"].sum())
)

print(
    "Кількість записів із визначеним сегментом:",
    int(tableau_df["behavior_segment"].notna().sum())
)

print(
    "Кількість унікальних сесій:",
    tableau_df["participant_session_id"].nunique()
)

display(tableau_df.head())

Кількість фактичних подій: 47657
Кількість записів після об'єднання: 47657
Кількість записів голосування: 34000
Кількість записів із визначеним сегментом: 46131
Кількість унікальних сесій: 1700


,event_id,page_view_id,event_timestamp,event_name,event_value,event_value_type,capture_side,visitor_id,user_id,user_type,...,session_dislikes,session_like_rate_pct,session_dislike_rate_pct,session_unique_movies,session_unique_genres,session_avg_movie_rating,session_avg_rating_liked,session_avg_rating_disliked,behavior_segment,genres_text
0,EV00000001,PV_LP_000001,2026-09-15 07:36:31+00:00,create_room_submitted,True,boolean,FRONTEND,visitor_000236,10236.0,AUTHENTICATED,...,10.0,50.0,50.0,20.0,15.0,7.064,7.361,6.767,Збалансовані,
1,EV00000008,PV_LP_000002,2026-10-02 07:13:57+00:00,guest_start_clicked,True,boolean,FRONTEND,visitor_000739,<NA>,GUEST,...,10.0,50.0,50.0,20.0,14.0,6.821,6.465,7.177,Збалансовані,
2,EV00000016,PV_LP_000003,2026-10-15 12:32:36+00:00,header_sign_in_clicked,True,boolean,FRONTEND,visitor_000966,<NA>,ANONYMOUS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
3,EV00000020,PV_LP_000004,2026-10-19 10:53:39+00:00,guest_start_clicked,True,boolean,FRONTEND,visitor_000967,<NA>,ANONYMOUS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
4,EV00000029,PV_LP_000005,2026-10-04 10:12:48+00:00,sign_up_clicked,True,boolean,FRONTEND,visitor_000968,<NA>,ANONYMOUS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,


In [5]:
# Перевіряємо унікальність event_id

if tableau_df["event_id"].duplicated().any():

    raise ValueError(
        "Після об'єднання виявлено дублікати event_id"
    )


# Перевіряємо, що кількість рядків не змінилася

if len(tableau_df) != len(activity_df):

    raise ValueError(
        "Після об'єднання змінилася кількість рядків"
    )


# Перевіряємо кількість голосів

if int(tableau_df["vote_count"].sum()) != len(vote_df):

    raise ValueError(
        "Кількість голосів не відповідає вихідній таблиці голосування"
    )


# Перевіряємо коректність позитивних і негативних голосів

if (
    tableau_df["like_count"].sum() +
    tableau_df["dislike_count"].sum()
    != len(vote_df)
):

    raise ValueError(
        "Сума Likes та Dislikes не відповідає кількості голосів"
    )


# Перевіряємо кількість сесій учасників

if len(session_summary) != (
    vote_df[["room_id", "participant_id"]]
    .drop_duplicates()
    .shape[0]
):

    raise ValueError(
        "Кількість сесій не відповідає кількості унікальних учасників кімнат"
    )


# Створюємо папку для Tableau

tableau_path = processed_path.parent / "data_tableau"

tableau_path.mkdir(
    parents=True,
    exist_ok=True
)


# Зберігаємо єдиний CSV-файл

output_file = tableau_path / "tableau_dataset.csv"

tableau_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)


# Перевіряємо збережений файл

saved_df = pd.read_csv(
    output_file,
    low_memory=False
)


# Виводимо підсумок

print("=" * 50)
print("TABLEAU DATASET SUMMARY")
print("=" * 50)

print("\nФайл успішно збережено:", output_file)

print(
    "Кількість рядків:",
    f"{len(saved_df):,}"
)

print(
    "Кількість колонок:",
    len(saved_df.columns)
)

print(
    "Кількість унікальних подій:",
    saved_df["event_id"].nunique()
)

print(
    "Кількість голосів:",
    f"{int(saved_df['vote_count'].sum()):,}"
)

print(
    "Кількість унікальних кімнат:",
    saved_df["room_id"].nunique()
)

print(
    "Кількість сесій учасників:",
    saved_df["participant_session_id"].nunique()
)

print(
    "Кількість унікальних фільмів:",
    saved_df["title_id"].nunique()
)

print(
    "Кількість сесій із визначеним сегментом:",
    saved_df["behavior_segment"].notna().sum()
)

print("\nПерші п'ять рядків:")

display(saved_df.head())

print("\nНазви колонок:")

for column in saved_df.columns:
    print(column)

TABLEAU DATASET SUMMARY

Файл успішно збережено: c:\Users\ACER\github\analytics\test_analytics\data_tableau\tableau_dataset.csv
Кількість рядків: 47,657
Кількість колонок: 49
Кількість унікальних подій: 47657
Кількість голосів: 34,000
Кількість унікальних кімнат: 500
Кількість сесій учасників: 1700
Кількість унікальних фільмів: 3445
Кількість сесій із визначеним сегментом: 46131

Перші п'ять рядків:


,event_id,page_view_id,event_timestamp,event_name,event_value,event_value_type,capture_side,visitor_id,user_id,user_type,...,session_dislikes,session_like_rate_pct,session_dislike_rate_pct,session_unique_movies,session_unique_genres,session_avg_movie_rating,session_avg_rating_liked,session_avg_rating_disliked,behavior_segment,genres_text
0,EV00000001,PV_LP_000001,2026-09-15 07:36:31+00:00,create_room_submitted,True,boolean,FRONTEND,visitor_000236,10236.0,AUTHENTICATED,...,10.0,50.0,50.0,20.0,15.0,7.064,7.361,6.767,Збалансовані,NaN
1,EV00000008,PV_LP_000002,2026-10-02 07:13:57+00:00,guest_start_clicked,True,boolean,FRONTEND,visitor_000739,NaN,GUEST,...,10.0,50.0,50.0,20.0,14.0,6.821,6.465,7.177,Збалансовані,NaN
2,EV00000016,PV_LP_000003,2026-10-15 12:32:36+00:00,header_sign_in_clicked,True,boolean,FRONTEND,visitor_000966,NaN,ANONYMOUS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,EV00000020,PV_LP_000004,2026-10-19 10:53:39+00:00,guest_start_clicked,True,boolean,FRONTEND,visitor_000967,NaN,ANONYMOUS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,EV00000029,PV_LP_000005,2026-10-04 10:12:48+00:00,sign_up_clicked,True,boolean,FRONTEND,visitor_000968,NaN,ANONYMOUS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Назви колонок:
event_id
page_view_id
event_timestamp
event_name
event_value
event_value_type
capture_side
visitor_id
user_id
user_type
participant_id
room_id
archive_session_id
title_id
room_title_id
interaction_method
is_liked
content_type
max_duration_minutes
participant_count
started_at
completed_at
card_position
title
genres_json
release_year
runtime_minutes
vote_average
likes_count
source_dataset
event_value_bool
is_vote_event
vote_type
vote_count
like_count
dislike_count
participant_session_id
session_total_votes
session_likes
session_dislikes
session_like_rate_pct
session_dislike_rate_pct
session_unique_movies
session_unique_genres
session_avg_movie_rating
session_avg_rating_liked
session_avg_rating_disliked
behavior_segment
genres_text
